# SynAgent Interactive Notebook

A short tour of SynAgent's Python interface: build the agent, chat with it, look at the tools it called, and validate a route.

`notebook.ipynb` does the same tour with the lower-level building blocks (`get_agent`, the workflow functions and pydantic-ai results).

## Setup

One import. The agent loads `.env` itself when it's built (pass `dotenv=False` to skip that), so the API keys don't need a separate step.

Jupyter is an optional extra. From the repository root, with `uv`:

```sh
uv run --extra jupyter jupyter lab notebook-examples/overview.ipynb
```

Without `uv`, install the package with the extra into a virtual environment, then start Jupyter from it:

```sh
python3 -m venv .venv
source .venv/bin/activate
pip install ".[jupyter]"
jupyter lab notebook-examples/overview.ipynb
```

In [ ]:
import synagent as sa

sa.specs()  # bundled specs: 'nogpu', 'retriever', 'synagent', 'validator'

['nogpu', 'retriever', 'synagent', 'validator']

## Build the agent

Specs are named rather than passed as file paths; a path still works for your own specs. Everything is a keyword, so `model` and `spec` can't be swapped by accident.

The agent's display shows what was loaded: the spec, the model, and the capabilities (tool groups) it can use.

In [ ]:
agent = sa.SynAgent(
    spec="nogpu",
    model="google:gemini-3.8-flash",
)
agent

SynAgent(spec='nogpu', model='anthropic:claude-opus-5-5')
  capabilities: AnalogueSearch, SynthesisValidation, Chemspace

## Chat

`agent.chat()` starts a conversation that keeps its own history. `await chat.ask(...)` streams tool calls and the reply into the cell while the agent works, like `synagent cli`, then returns a `Reply` that renders as Markdown.

Pass `show_tools=False` for just the final answer. A streamed reply isn't displayed a second time; `reply.show()` renders it again.

In [ ]:
chat = agent.chat()

await chat.ask(
    "Is CC(=O)Oc1ccccc1C(=O)O a valid SMILES? If so, is it a building block?"
)

╭───────────────────────────────────────────── Tool: load_capability ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "id": "synthesis-validation"                                                                                  │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Tool: load_capability ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "id": "analogue-search"                                                                                       │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Result: load_capability ────────────────────────────────────────────╮
│ {"instructions":"Use the analogue searching capability to find building blocks that fit reaction templates. Use │
│ check_building_blocks and check_templates to confirm that a route only uses building blocks and templates from  │
│ the libraries."}                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Result: load_capability ────────────────────────────────────────────╮
│ {"instructions":"Validate a synthesis route with these tools. Report exactly what the tools return. If you edit │
│ a SMILES or template, validate the edited version; a step passes only on the version that was                   │
│ validated.\n\nProcedure:\n1. Collect every molecule in the route: building blocks, intermediates, and the       │
│ target. Check them in a single validate_smiles call.\n2. Check all reaction templates in a single               │
│ validate_reaction_smarts call.\n3. For each step, call validate_products with the template…                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Tool: validate_smiles ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "smiles": [                                                                                                   │
│     "CC(=O)Oc1ccccc1C(=O)O"                                                                                     │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── Tool: check_building_blocks ──────────────────────────────────────────╮
│ {                                                                                                               │
│   "smiles": [                                                                                                   │
│     "CC(=O)Oc1ccccc1C(=O)O"                                                                                     │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Result: check_building_blocks ─────────────────────────────────────────╮
│ {"CC(=O)Oc1ccccc1C(=O)O":true}                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Result: validate_smiles ────────────────────────────────────────────╮
│ {"CC(=O)Oc1ccccc1C(=O)O":true}                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

I need tools for validation and building block checks. Let me load the relevant capabilities.Yes to both.          

 1 Valid SMILES: CC(=O)Oc1ccccc1C(=O)O passed validation. It's acetylsalicylic acid (aspirin), 2-acetoxybenzoic    
   acid.                                                                                                           
 2 Building block: Yes. An exact-match check found it in the building block library.                               

Because it's already a building block, it doesn't need a synthesis route, so I stopped there. You can use it       
directly as a starting material. For example, its carboxylic acid could be used for amide or ester formation.

Follow-ups go to the same `chat`, with no history to pass around.

In [ ]:
reply = await chat.ask(
    "Propose a one-step route to it from building blocks in the library."
)

╭───────────────────────────────────────────── Tool: validate_smiles ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "smiles": [                                                                                                   │
│     "O=C(O)c1ccccc1O",                                                                                          │
│     "CC(=O)Cl",                                                                                                 │
│     "CC(=O)OC(C)=O",                                                                                            │
│     "CC(=O)O"                                                                                                   │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── Tool: validate_reaction_smarts ─────────────────────────────────────────╮
│ {                                                                                                               │
│   "reaction_smarts": [                                                                                          │
│     "[C:3](=[O:4])[Cl].[OH:2]>>[O:2][C:3]=[O:4]",                                                               │
│     "[CH3:5][C:3](=[O:4])[O][C](=O)[CH3].[OH:2]>>[O:2][C:3](=[O:4])[CH3:5]",                                    │
│     "[C:3](=[O:4])[OH].[OH:2]>>[O:2][C:3]=[O:4]"                                                                │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── Result: validate_reaction_smarts ────────────────────────────────────────╮
│ {"[C:3](=[O:4])[Cl].[c:1][OH:2]>>[c:1][O:2][C:3]=[O:4]":true,"[CH3:5][C:3](=[O:4])[O][C](=O)[CH3].[c:1][OH:2]>> │
│ [c:1][O:2][C:3](=[O:4])[CH3:5]":true,"[C:3](=[O:4])[OH].[c:1][OH:2]>>[c:1][O:2][C:3]=[O:4]":true}               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Result: validate_smiles ────────────────────────────────────────────╮
│ {"O=C(O)c1ccccc1O":true,"CC(=O)Cl":true,"CC(=O)OC(C)=O":true,"CC(=O)O":true}                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Tool: check_templates ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "reaction_templates": [                                                                                       │
│     "[C:3](=[O:4])[Cl].[OH:2]>>[O:2][C:3]=[O:4]",                                                               │
│     "[CH3:5][C:3](=[O:4])[O][C](=O)[CH3].[OH:2]>>[O:2][C:3](=[O:4])[CH3:5]",                                    │
│     "[C:3](=[O:4])[OH].[OH:2]>>[O:2][C:3]=[O:4]"                                                                │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── Tool: check_building_blocks ──────────────────────────────────────────╮
│ {                                                                                                               │
│   "smiles": [                                                                                                   │
│     "O=C(O)c1ccccc1O",                                                                                          │
│     "CC(=O)Cl",                                                                                                 │
│     "CC(=O)OC(C)=O",                                                                                            │
│     "CC(=O)O"                                                                                                   │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (15 outputs left out)

╭───────────────────────────────────────── Result: check_building_blocks ─────────────────────────────────────────╮
│ {"COC(=O)c1ccccc1OC(C)=O":false,"CCOC(=O)c1ccccc1OC(C)=O":false,"CC(=O)Oc1ccccc1C(=O)OC(C)(C)C":false,"CC(=O)Oc │
│ 1ccccc1C(=O)OCc1ccccc1":false}                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Tool: search_building_blocks ──────────────────────────────────────────╮
│ {                                                                                                               │
│   "smiles": [                                                                                                   │
│     "COC(=O)c1ccccc1OC(C)=O"                                                                                    │
│   ],                                                                                                            │
│   "threshold": 0.6,                                                                                             │
│   "max_results": 20                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── Result: search_building_blocks ─────────────────────────────────────────╮
│ {"COC(=O)c1ccccc1OC(C)=O":["CCOC(=O)c1ccc(O)c(C=O)c1","COC(=O)c1ccc(OC)c(C=O)c1","COC(=O)c1cc(C=O)ccc1OC","COC( │
│ =O)c1cc(CO)ccc1OC","COCc1cc(C(=O)O)ccc1OC","COC(=O)c1ccc(CO)c(OC)c1","COC(=O)c1ccc(C=O)c(OC)c1","COC(=O)c1ccc(C │
│ =O)cc1OC","COC(=O)COc1ccccc1C(=O)O","COC(=O)c1ccccc1OCC(=O)O","COCc1ccc(C(=O)O)cc1OC","CCCOc1cc(C)ccc1C(=O)O"," │
│ CC(C)COc1ccccc1C(=O)O","CC(=O)Oc1ccccc1C(=O)O","C#Cc1ccc(OC)c(C(=O)OC)c1","COC(=O)c1ccc(C)c(OCF)c1","C#Cc1ccc(C │
│ (=O)OC)c(OC)c1","CCC(C)Oc1ccccc1C(=O)O","COC(=O)c1ccc(O)…                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

I couldn't find a verified one-step route to aspirin using only library templates and building blocks. The usual   
chemistry needs a reaction that the template library doesn't have.                                                 

What I tried                                                                                                       

                                                                                                                   
 Disconnection                 Precursors                   Precursors in library?   Template in library?          
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
 1. React salicylic acid's OH  O=C(O)c1ccccc1O + CC(=O)Cl   Both yes                 No                            
 with acetyl chloride                                                                ([C:3](=[O:4])[Cl].[c:1][OH:… 
 2. React salicylic acid's OH  O=C(O)c1ccccc1O +            Acetic anhydride is not  No                            
 with acetic anhydride (the    CC(=O)OC(C)=O                                                                       
 textbook route)                                                                                                   
 3. Join acetic acid directly  O=C(O)c1ccccc1O + CC(=O)O    Both yes                 No                            
 to salicylic acid's OH                                                                                            
 4. Cleave the ester group of  Methyl, ethyl, tert-butyl    None of them             Yes                           
 a 2-acetoxybenzoate ester     or benzyl 2-acetoxybenzoate                           ([C:1](=[O:2])[O:3][#6]>>[C:… 
                                                                                                                   

 • Disconnections 1–3: The library has no template for adding an acyl group to a phenol (an aromatic OH). The      
   closest library template joins an acid to an OH, but only when the OH is on a non-aromatic carbon. I ran it on  
   acetic acid + salicylic acid and it gave no products. Searches at lower similarity didn't turn up a suitable    
   template either.                                                                                                
 • Disconnection 4: The template is in the library, but none of the four esters is a building block. A similarity  
   search didn't find a replacement. Even with a suitable ester, the same template would probably also cleave the  
   acetate, so it would be unselective.                                                                            

Conclusion                                                                                                         

None of the four disconnections gives a one-step route with every template and building block in the library, so I 
can't call any of them verified.                                                                                   

In practice, the standard route is disconnection 1 or 2: salicylic acid with acetyl chloride or acetic anhydride,  
usually with an acid catalyst or pyridine. Salicylic acid and acetyl chloride are both in the library; only the    
template is missing. Aspirin itself is a library building block, so it can be used directly as a starting material.

If you want, I can check prices and availability on Chemspace, or look at multi-step options. Those are unlikely to
help, though, since the missing template would still be the problem.

For a one-off question, `agent.ask(...)` runs a single turn with no history.

In [ ]:
await agent.ask("Validate the SMILES c1ccccc1N", show_tools=False)

## Inspect the run

The `Reply` holds the text, the tool calls (each with its arguments and result), and usage. `reply.tool_calls` displays as a table. The underlying pydantic-ai result stays available as `reply.result` for anything else.

In [ ]:
reply.tool_calls  # one row per call: tool, args, result

In [ ]:
reply.tool_calls[0].args, reply.tool_calls[0].result

In [ ]:
reply.usage  # requests=7, tool_calls=10, tokens in/out, cost

In [ ]:
chat.history  # every turn so far; chat.reset() starts over, like /clear in the REPL

## Routes

A route is a list of steps, each `(template, reactants, product)`. Step numbers come from the order, and building blocks are the reactants that no earlier step produces, so you don't have to list them twice. In a notebook a route draws as a reaction scheme with RDKit.

`sa.Route.from_json(...)` reads the SynLlama format that `synagent run` emits.

In [ ]:
route = sa.Route.from_steps(
    [
        (
            "[OH:1][c:2].[Cl:3][C:4]=[O:5]>>[O:1]([c:2])[C:4]=[O:5]",
            ["Oc1ccccc1C(=O)O", "CC(=O)Cl"],
            "CC(=O)Oc1ccccc1C(=O)O",
        ),
    ]
)
route  # draws each step; route.building_blocks -> ['Oc1ccccc1C(=O)O', 'CC(=O)Cl']

## Workflows

The workflows from `synagent run` are agent methods. They're async like everything else here, so no `asyncio.to_thread` is needed.

| Method | Returns | Needs |
| --- | --- | --- |
| `agent.validate(route)` | `Report` | any spec with validation tools |
| `agent.correct(report)` | `Report` | any spec with validation tools |
| `agent.generate(request)` | `Route` | SynLlama (default spec) |
| `agent.pipeline(request, max_iter=4)` | `Report` | SynLlama (default spec) |

Calling one the spec can't run fails right away with a message naming the missing capability, before any model request. With the `nogpu` spec, for example, `agent.generate(...)` raises `UserError: generate() needs the Generation capability`.

In [ ]:
report = await agent.validate(route)
report  # summary line and a table of steps

In [ ]:
report.passed  # all_building_blocks_valid and all_reactions_passed

In [ ]:
report.failed_steps  # [] here; otherwise the steps with their failure_mode

In [ ]:
report.to_polars()  # one row per step, for filtering and joins

If a step fails, hand the report back to fix it:

```python
if not report.passed:
    report = await agent.correct(report)
```

## Tracing (optional)

`trace=True` writes the JSONL trace to the default location; pass a path to choose one. The agent reports where the file went. `synagent trace export` turns these files into a fine-tuning dataset.

In [ ]:
traced = sa.SynAgent(spec="nogpu", trace=True)

await traced.ask("Validate the SMILES c1ccccc1N", show_tools=False)
traced.trace_path

## Scripts

Outside a notebook there's no running event loop. Every async method has a blocking twin with a `_sync` suffix:

```python
import synagent as sa

agent = sa.SynAgent(spec="nogpu")
route = sa.Route.from_json(open("route.json").read())
report = agent.validate_sync(route)
print(report.passed)
```